In [3]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai


In [4]:
# Instala el conector nuevo de Gemini para LlamaIndex
# Configura tu clave de Gemini
import os
from getpass import getpass
os.environ["GOOGLE_API_KEY"] = getpass("Introduce tu clave de Gemini: ")

Introduce tu clave de Gemini: ··········


In [5]:
# Importaciones
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings

# Asignar el modelo de lenguaje
Settings.llm = GoogleGenAI(model="gemini-2.5-pro")

# Asignar el modelo de embeddings de Google, que se encarga de convertir texto a embeddings
Settings.embed_model = GoogleGenAIEmbedding(model_name="text-embedding-004", api_key=os.environ["GOOGLE_API_KEY"])

# Crear carpeta y documento de ejemplo
os.makedirs("/content/docs", exist_ok=True)
with open("/content/docs/info.txt", "w", encoding="utf-8") as f:
    f.write('''
El Barcelona Supercomputing Center – Centro Nacional de Supercomputación (BSC-CNS) es una institución líder en investigación científica y tecnológica. Su misión principal es investigar, desarrollar y gestionar tecnologías de supercomputación con el objetivo de ofrecer soporte a la comunidad científica y al sector industrial.

El BSC trabaja en tres áreas clave: inteligencia artificial (IA), bioinformática y supercomputación avanzada. En el campo de la IA, el centro desarrolla algoritmos de aprendizaje profundo, modelos de lenguaje y herramientas de análisis masivo de datos aplicados a la industria, la salud y la sostenibilidad.

En el área de bioinformática, el BSC impulsa proyectos que integran la biología y la computación de alto rendimiento, con el fin de comprender mejor el genoma humano, acelerar el descubrimiento de fármacos y analizar grandes volúmenes de datos biomédicos.

La tercera gran línea es la supercomputación, donde el centro opera MareNostrum, uno de los superordenadores más potentes de Europa. Este sistema permite realizar simulaciones a gran escala para campos tan diversos como el cambio climático, la física cuántica, la ingeniería o la cosmología.

    ''')

# Cargar los documentos
docs = SimpleDirectoryReader("/content/docs").load_data()

# Crear índice vectorial utilizando los embeddings configurados
index = VectorStoreIndex.from_documents(docs)

# Crear motor de consulta
query_engine = index.as_query_engine()

# Hacer una pregunta al documento
answer = await query_engine.aquery("¿En qué áreas trabaja el BSC?")
print(answer)

El Barcelona Supercomputing Center (BSC) trabaja en tres áreas clave:

1.  **Inteligencia artificial (IA):** Desarrolla algoritmos de aprendizaje profundo, modelos de lenguaje y herramientas de análisis masivo de datos para sectores como la industria, la salud y la sostenibilidad.
2.  **Bioinformática:** Impulsa proyectos que combinan biología y computación de alto rendimiento para comprender el genoma humano, acelerar el descubrimiento de fármacos y analizar datos biomédicos.
3.  **Supercomputación avanzada:** Opera el superordenador MareNostrum, que permite realizar simulaciones a gran escala para campos como el cambio climático, la física cuántica, la ingeniería y la cosmología.


In [6]:
!pip install -U langchain-google-genai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's depe

In [1]:
# Importamos librerías necesarias ---
from getpass import getpass
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Solicitamos la API Key de forma segura ---
# No se mostrará en pantalla ni quedará guardada en texto plano.
api_key = getpass("Introduce tu clave de Google Gemini (no se mostrará): ")

# Guardamos la clave como variable de entorno temporal
os.environ["GOOGLE_API_KEY"] = api_key

# Configuramos el modelo Gemini
# Puedes cambiar por "gemini-1.5-pro-latest" si tu cuenta lo soporta.
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")

# Definimos el tipo de estado que circulará por el grafo
class State(TypedDict):
    question: str
    answer: str

# Definimos el nodo que genera la respuesta
def responder(state: State) -> dict:
    """
    Usa el modelo Gemini para responder a la pregunta contenida en el estado.
    """
    output = llm.invoke(state["question"])
    return {"answer": output.content}

# Construimos el grafo básico de tipado
# Esto va a definir el flujo de la información entre nodos
graph = StateGraph(State)
# Primer nodo responder, el cual ejecutará esa función
graph.add_node("responder", responder)
# Conecto responder con el nodo inicial START
graph.add_edge(START, "responder")
# Ahora con el nodo final end
graph.add_edge("responder", END)

# Compilamos el grafo
app = graph.compile()

# Ejecutamos el flujo con una pregunta
estado_inicial = {"question": "Explícame en una frase qué es un agente de IA", "answer": ""}
resultado = app.invoke(estado_inicial)

# Mostramos la respuesta
print("\nRespuesta del modelo:")
print(resultado["answer"])

Introduce tu clave de Google Gemini (no se mostrará): ··········

💬 Respuesta del modelo:
Es una entidad de software o hardware que percibe su entorno, razona y ejecuta acciones de forma autónoma para alcanzar objetivos específicos.
